# Citiz — 4. Accès à 5 minutes et population desservie (Gold)
Runtime Python. À exécuter après Silver ; **aucun téléchargement ni recalcul Bronze/Silver**.
**Objectif 1** : pour chaque carreau habité de France, le nombre d'options de mobilité pondérées atteintes
à 400 m (5 min à 4,8 km/h), agrégé par commune et rapporté à l'habitant.
**Objectif 2** : pour chaque arrêt ou station, la population desservie et celle dont c'est la seule option.
Distance à vol d'oiseau (V1) : **borne haute** (à Grenoble, la V1 surestime les options de 80 % par rapport
à la marche réelle). Silver ne donne ni le mode (bus, tram, train) ni la fréquence des arrêts : chaque arrêt
vaut 1. Les stations GBFS sont lues en Bronze (même run), car `dim_station` ne garde que l'autopartage.
Secret statistique : carreaux imputés jamais affichés seuls ; populations desservies arrondies à 5.
Profil social des carreaux (Filosofi, table Bronze `carreaux_filosofi`) : niveau de vie, ménages pauvres, âges.

In [ ]:
%pip install -q "deltalake>=1,<2" scipy requests

In [ ]:
BRONZE = 'bronze'
SILVER = 'silver'
GOLD = 'gold'
SCHEMA = 'dbo'
LOCAL_ROOT = None
RAYON_M = 400            # 5 minutes à 4,8 km/h
FUSION_ARRETS_M = 50     # quais d'un même arrêt et doublons entre flux : une seule option
FUSION_STATIONS_M = 20   # même station publiée par deux flux GBFS
POIDS = {'tc': 1.0, 'citiz': 1.5, 'autopartage': 1.5, 'velo': 1.0, 'trottinette': 0.5}
POIDS_FREE_FLOATING_PAR_MODE = 0.5   # opérateur Dott/Voi présent à 400 m : 0,5 par mode (vélo, trottinette)
MARGUERITE_URL = ('https://data.nantesmetropole.fr/api/explore/v2.1/catalog/datasets/'
                  '244400404_stations-autopartage-naolib-nantes-metropole/exports/geojson')
SEUIL_DIFFUSION = 20     # population desservie masquée en dessous (≈ 11 ménages), arrondie à 5 sinon
ZONES_FOCUS = {'Nantes Métropole': '244400404', 'Grenoble-Alpes-Métropole': '200040715'}   # pages zoom du rapport

In [ ]:
import os, re, json, unicodedata, uuid, datetime as dt
import numpy as np
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from scipy.spatial import cKDTree
RUN_ID = str(uuid.uuid4())
STATUS = []

def nu():
    import notebookutils
    return notebookutils

def root(lakehouse):
    if LOCAL_ROOT:
        return f"{LOCAL_ROOT}/{lakehouse}", None
    info = nu().lakehouse.get(lakehouse)
    if not isinstance(info, dict):
        info = json.loads(json.dumps(info, default=lambda x: x.__dict__))
    props = info.get("properties", {})
    path = props.get("abfsPath")
    if not path:
        raise RuntimeError(f"abfsPath absent pour {lakehouse}; contrôler notebookutils.lakehouse.get")
    return path.rstrip("/"), {"bearer_token": nu().credentials.getToken("storage"), "use_fabric_endpoint":"true"}

def table_path(lakehouse, name):
    p,o = root(lakehouse)
    return f"{p}/Tables/{SCHEMA + '/' if SCHEMA else ''}{name}",o

def read_table(lakehouse, name, required=False, run_id=None, columns=None):
    p,o = table_path(lakehouse,name)
    try:
        dt_=DeltaTable(p,storage_options=o)
        if columns: columns=[c for c in columns+['_run_id'] if c in {f.name for f in dt_.schema().fields}]
        t=dt_.to_pandas(columns=columns)
    except Exception as e:
        if required: raise RuntimeError(f"Table obligatoire illisible: {lakehouse}.{name}") from e
        print(f"Absente/illisible: {lakehouse}.{name}: {str(e)[:120]}")
        return pd.DataFrame()
    if run_id is not None:
        if '_run_id' not in t: return pd.DataFrame()
        t=t[t['_run_id']==run_id].copy()
    return t.drop(columns=['_run_id'],errors='ignore')

def write_table(lakehouse,name,df):
    df=df.reset_index(drop=True).copy()
    df['_run_id']=RUN_ID
    # Types explicites même pour colonnes entièrement nulles / tables vides.
    a=pa.Table.from_pandas(df,preserve_index=False)
    a=pa.Table.from_arrays([a[c].cast(pa.string()) if pa.types.is_null(a[c].type) else a[c]
                            for c in a.column_names],names=a.column_names)
    p,o=table_path(lakehouse,name)
    if LOCAL_ROOT: os.makedirs(os.path.dirname(p),exist_ok=True)
    write_deltalake(p,a,mode='overwrite',schema_mode='overwrite',storage_options=o)
    STATUS.append({'table':name,'lignes':len(df),'run_id':RUN_ID})
    print(f"{lakehouse}.{name}: {len(df):,} lignes")

def nnum(s):
    return pd.to_numeric(s.astype('string').str.replace(' ','',regex=False)
          .str.replace(' ','',regex=False).str.replace(',','.',regex=False),errors='coerce')

def valid_points(df):
    d=df.copy()
    d['lat']=nnum(d['lat']); d['lon']=nnum(d['lon'])
    return d[d.lat.between(-90,90)&d.lon.between(-180,180)&~((d.lat==0)&(d.lon==0))].copy()

def xyz(lon,lat):
    a,b=np.radians(np.asarray(lon,dtype=float)),np.radians(np.asarray(lat,dtype=float))
    return np.column_stack([np.cos(b)*np.cos(a),np.cos(b)*np.sin(a),np.sin(b)])

def chord(metres):
    return 2*np.sin(metres/(2*6371008.8))

## 1 — Lecture du run Silver
Les tables écrites ici portent l'identifiant du run Silver : l'optimisation les relit avec lui.

In [ ]:
run=read_table(SILVER,'pipeline_run',True)
if len(run)!=1 or run.iloc[0]['status']!='complete': raise ValueError('Silver non validé')
RUN_ID=run.iloc[0]['run_id']
cells=read_table(SILVER,'fact_carreau_population',True,RUN_ID)
stops=read_table(SILVER,'dim_arret',True,RUN_ID)
pop=read_table(SILVER,'fact_population',True,RUN_ID)
com=read_table(SILVER,'dim_commune',True,RUN_ID,columns=['commune_code','nom','code_dept','code_region','population_api','lat','lon'])
gbfs=read_table(BRONZE,'gbfs_stations',True,RUN_ID)
filo=read_table(BRONZE,'carreaux_filosofi',False,RUN_ID)   # absente : profil social inconnu, calcul inchangé
print(f'{len(cells):,} carreaux, {len(stops):,} arrêts, {len(gbfs):,} stations GBFS')
import requests
focus={}
for nom_zone,epci in ZONES_FOCUS.items():
    r=requests.get(f'https://geo.api.gouv.fr/epcis/{epci}/communes',params={'fields':'code'},timeout=60); r.raise_for_status()
    focus.update({c['code']:nom_zone for c in r.json()})

## 2 — Population calée sur le recensement 2023
`pop_calee = population(carreau) × population 2023(commune) / Σ population des carreaux de la commune`.
Filosofi ne compte pas les personnes en logement collectif (étudiants, foyers) : −10 % à Grenoble.
Commune absente du recensement 2023 de Silver (Paris) : population légale `population_api` en repli.

In [ ]:
cells=valid_points(cells[cells.population>0])
p23=pop[pop.annee==2023].groupby('commune_code').population.sum()
total=p23.combine_first(com.set_index('commune_code').population_api.astype(float).where(lambda s:s>0))
ratio=cells.commune_code.map(total)/cells.groupby('commune_code').population.transform('sum')
cells['pop_calee']=(cells.population*ratio).fillna(cells.population)
cells=cells.reset_index(drop=True)
print(f"Population calée : {cells.pop_calee.sum():,.0f} (Filosofi brut : {cells.population.sum():,.0f})")

## 3 — Offre de mobilité : une ligne par option
- **Transports en commun** : arrêts GTFS fusionnés à 50 m (quais d'un même arrêt, doublons entre flux).
- **GBFS** : autopartage (Citiz ou flux à voitures), zones de free-floating (Dott, Voi, Pony…),
  sinon station de vélos (trottinettes seules : poids 0,5).
- **Marguerite** (Nantes, open data de Nantes Métropole, hors Citiz déjà dans le GBFS).

In [ ]:
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import connected_components

def fuse(points, metres, keys):
    """Fusionne les points d'un même groupe distants de moins de `metres` (agrégation à lien simple)."""
    parts=[]
    for _,g in points.groupby(keys,sort=True,dropna=False):
        g=g.reset_index(drop=True)
        pairs=cKDTree(xyz(g.lon,g.lat)).query_pairs(chord(metres),output_type='ndarray')
        n=len(g); graph=coo_matrix((np.ones(len(pairs)),(pairs[:,0],pairs[:,1])),shape=(n,n))
        g['_cl']=connected_components(graph,directed=False)[1]
        agg={c:'first' for c in g.columns if c not in ('_cl','lat','lon')}
        agg.update({'lat':'mean','lon':'mean'})
        m=g.groupby('_cl',sort=False).agg(agg); m['n_fusionnes']=g.groupby('_cl',sort=False).size()
        parts.append(m.reset_index(drop=True))
    return pd.concat(parts,ignore_index=True)

tc=valid_points(stops)[['arret_id','nom','lat','lon','commune_code']].rename(columns={'arret_id':'point_id'})
tc=fuse(tc.assign(famille='tc',operateur='GTFS'),FUSION_ARRETS_M,['famille'])
tc['poids']=POIDS['tc']

g=valid_points(gbfs).copy()
types=(g.all_type_ids.fillna('')+' '+g.station_type_ids.fillna('')).str.lower()
voiture=g.is_citiz.fillna(False).astype(bool)|g.car_type_ids.fillna('[]').ne('[]')|g.station_type_ids.fillna('[]').ne('[]')
zone=~voiture&(g.operator.fillna('').str.contains(r'dott|voi\b|voi technology|pony|lime|tier|bird|bolt|micromob',case=False)
               |types.str.contains(r'dott_|voi_|pony'))
velo_mode=types.str.contains(r'bike|bicycle|velo|mechanical|electrical|titibike')
trott_mode=types.str.contains(r'scooter|trott|knot|x2')
g['famille']=np.select([g.is_citiz.fillna(False).astype(bool),voiture,zone,~velo_mode&trott_mode],
                       ['citiz','autopartage','free_floating','trottinette'],default='velo')
modes=(velo_mode.astype(int)+trott_mode.astype(int)).clip(lower=1)
g['poids']=np.where(g.famille.eq('free_floating'),POIDS_FREE_FLOATING_PAR_MODE*modes,g.famille.map(POIDS))
g=g.rename(columns={'name':'nom','operator':'operateur'})
g['point_id']='gbfs:'+g.feed_id.astype(str)+':'+g.station_id.astype(str)
g=fuse(g[['point_id','nom','operateur','famille','poids','lat','lon','capacity']],FUSION_STATIONS_M,['famille'])

marg=pd.DataFrame()
try:
    import requests
    r=requests.get(MARGUERITE_URL,timeout=60); r.raise_for_status()
    feats=r.json()['features']
    marg=pd.DataFrame([{**f['properties'],'lon':f['geometry']['coordinates'][0],'lat':f['geometry']['coordinates'][1]}
                       for f in feats if f.get('geometry')])
    marg=marg[~marg.exploitant.fillna('').str.contains('citiz',case=False)]
    marg=pd.DataFrame({'point_id':'marguerite:'+marg.gid.astype(str),'nom':marg.nom,'operateur':'Marguerite',
                       'famille':'autopartage','poids':POIDS['autopartage'],'lat':marg.lat,'lon':marg.lon,
                       'capacity':nnum(marg.capacite),'n_fusionnes':1})
    STATUS.append({'table':'source_marguerite','lignes':len(marg),'run_id':RUN_ID})
except Exception as e:
    # Source absente = inconnue : l'autopartage hors GBFS de Nantes manque, sans faire planter le calcul.
    print('Marguerite indisponible :',str(e)[:150])
    STATUS.append({'table':'source_marguerite','lignes':0,'run_id':f'indisponible: {str(e)[:100]}'})

offer=pd.concat([tc,g,marg],ignore_index=True)
offer=offer[offer.poids.notna()&offer.poids.gt(0)].reset_index(drop=True)
print(offer.groupby('famille').agg(points=('point_id','size'),poids=('poids','sum')).to_string())

## 4 — Couverture à 400 m et options pondérées par carreau
`C[i, j] = 1` si le point d'offre j est à moins de 400 m du centre du carreau i.
`options_ponderees(i) = Σ_j C[i, j] × poids(j)`, l'indicateur de la carte (objectif 1).
Exception : le free-floating compte une fois par opérateur (0,5 par mode) dès qu'une zone est à 400 m.

In [ ]:
def coverage(q, refs, metres, step=200000):
    tree=cKDTree(xyz(refs.lon,refs.lat)); rows=[]; cols=[]
    for s in range(0,len(q),step):
        pairs=cKDTree(xyz(q.lon.values[s:s+step],q.lat.values[s:s+step])).sparse_distance_matrix(
            tree,chord(metres),output_type='ndarray')
        rows.append(pairs['i'].astype(np.int64)+s); cols.append(pairs['j'].astype(np.int64))
    r,c=np.concatenate(rows),np.concatenate(cols)
    return csr_matrix((np.ones(len(r),dtype=np.float32),(r,c)),shape=(len(q),len(refs)))

C=coverage(cells,offer,RAYON_M)
w=offer.poids.to_numpy(float)
fam=offer.famille.to_numpy()
ff=fam=='free_floating'
cells['options_ponderees']=C[:,~ff]@w[~ff]
# Free-floating : présence par opérateur (0,5 par mode), pas une option par zone (26 000 zones Dott à Paris).
for op in offer.operateur[ff].dropna().unique():
    idx=np.flatnonzero(ff&offer.operateur.eq(op).to_numpy())
    cells['options_ponderees']+=C[:,idx].multiply(w[idx]).max(axis=1).toarray().ravel()
cells['nb_points']=np.diff(C.indptr)
for f in ['tc','citiz','autopartage','velo','trottinette','free_floating']:
    cells[f'couvert_{f}']=(C[:,np.flatnonzero(fam==f)].getnnz(axis=1)>0) if (fam==f).any() else False
cells['couvert_autopartage']=cells.couvert_autopartage|cells.couvert_citiz   # toute voiture partagée
cells['sans_option']=cells.nb_points.eq(0)
cells['affichable']=cells.population_imputee.eq(0)

def classe(s):
    return pd.cut(s,[-0.1,0,2,5,10,20,np.inf],labels=['0','moins de 2','2 à 5','5 à 10','10 à 20','20 et plus']).astype(str)

cells['classe_acces']=classe(cells.options_ponderees)
cells['zone_focus']=cells.commune_code.map(focus)
# Profil social (Filosofi 2021). Valeurs imputées pour les carreaux de moins de 11 ménages : jamais affichées seules.
SOCIO=['ind','men','men_pauv','ind_snv','ind_18_24','ind_65_79','ind_80p']
if len(filo):
    cells=cells.merge(filo[['carreau_id']+SOCIO].rename(columns={'ind':'ind_filosofi'}),on='carreau_id',how='left')
else:
    cells=cells.assign(**{c:np.nan for c in ['ind_filosofi']+SOCIO[1:]})
cells['niveau_vie_moyen']=cells.ind_snv/cells.ind_filosofi.where(cells.ind_filosofi>0)
cells['part_menages_pauvres']=cells.men_pauv/cells.men.where(cells.men>0)
cells['part_18_24']=cells.ind_18_24/cells.ind_filosofi.where(cells.ind_filosofi>0)
cells['part_65_plus']=(cells.ind_65_79+cells.ind_80p)/cells.ind_filosofi.where(cells.ind_filosofi>0)
assert cells.carreau_id.is_unique and (cells.options_ponderees>=0).all()
assert (cells.sans_option==(cells.options_ponderees==0)).all()
print(f"Carreaux : {len(cells):,}, paires couvertes : {C.nnz:,}")

## 5 — Population desservie par point d'offre (objectif 2)
`pop_desservie(j) = Σ_i C[i, j] × pop_calee(i)` ; `pop_seule_option(j)` : habitants dont j est la seule option.
Les bassins se recouvrent : ne jamais sommer `pop_desservie` entre points.

In [ ]:
P=cells.pop_calee.to_numpy(float)
only=cells.nb_points.eq(1).to_numpy()
offer['pop_desservie']=C.T@P
offer['pop_seule_option']=C.T@(P*only)
# Commune du point : celle du carreau habité le plus proche (sans objet loin de tout carreau).
d,i=cKDTree(xyz(cells.lon,cells.lat)).query(xyz(offer.lon,offer.lat),k=1)
near=d<=chord(2*RAYON_M)
offer['commune_code']=offer.commune_code.where(offer.commune_code.notna(),pd.Series(np.where(near,cells.commune_code.to_numpy()[i],None)))

def diffusable(s):
    return (np.round(s/5)*5).where(s>=SEUIL_DIFFUSION)

publie=offer.assign(pop_desservie=diffusable(offer.pop_desservie),pop_seule_option=diffusable(offer.pop_seule_option))
publie=publie.merge(com[['commune_code','nom','code_dept']].rename(columns={'nom':'commune'}),on='commune_code',how='left')
rows=[]
for f in sorted(offer.famille.unique()):
    covered=C[:,np.flatnonzero(fam==f)].getnnz(axis=1)>0
    rows.append({'famille':f,'nb_points':int((fam==f).sum()),'pop_couverte':float(np.round(P[covered].sum()/5)*5),
                 'part_pop_couverte':float(P[covered].sum()/P.sum())})
famille=pd.DataFrame(rows).sort_values('pop_couverte',ascending=False)
print(famille.to_string(index=False))

## 6 — Agrégats par commune (carte France) et écriture
`options_par_habitant(commune) = Σ pop_calee × options_ponderees / Σ pop_calee`.
La carte utilise `gold_acces_commune` (centre de la commune, couleur par options par habitant) ;
`gold_acces_carreau` ne s'affiche que filtré sur `affichable` et sur une zone (Nantes, Grenoble).

In [ ]:
cells['pop_x_options']=cells.pop_calee*cells.options_ponderees
agg={'pop_calee':'sum','pop_x_options':'sum','carreau_id':'size'}
for f in ['sans_option','couvert_tc','couvert_citiz','couvert_autopartage','couvert_velo']:
    cells[f'_p_{f}']=cells.pop_calee*cells[f]; agg[f'_p_{f}']='sum'
k=cells.groupby('commune_code').agg(agg).rename(columns={'carreau_id':'nb_carreaux'})
k['options_par_habitant']=k.pop_x_options/k.pop_calee
for f in ['sans_option','couvert_tc','couvert_citiz','couvert_autopartage','couvert_velo']:
    k['part_pop_'+f.replace('couvert_','')]=k.pop(f'_p_{f}')/k.pop_calee
k=k.drop(columns='pop_x_options').reset_index()
so=cells.groupby('commune_code')[['ind_filosofi','men','men_pauv','ind_snv','ind_18_24','ind_65_79','ind_80p']].sum(min_count=1)
k=k.merge(pd.DataFrame({'niveau_vie_moyen':so.ind_snv/so.ind_filosofi,'part_menages_pauvres':so.men_pauv/so.men,
                        'part_18_24':so.ind_18_24/so.ind_filosofi,'part_65_plus':(so.ind_65_79+so.ind_80p)/so.ind_filosofi}),
          on='commune_code',how='left')
k['pop_calee']=np.round(k.pop_calee/5)*5
k['classe_acces']=classe(k.options_par_habitant)
k=k.merge(com,on='commune_code',how='left')
k['zone_focus']=k.commune_code.map(focus)
publie['zone_focus']=publie.commune_code.map(focus)
cells=cells.drop(columns=[c for c in cells if c.startswith('_p_')]+['pop_x_options','ind_filosofi']+SOCIO[1:])

write_table(GOLD,'gold_acces_carreau',cells)
write_table(GOLD,'gold_acces_commune',k)
write_table(GOLD,'gold_offre_desservie',publie)
write_table(GOLD,'gold_desserte_famille',famille)
params={'rayon_m':RAYON_M,'fusion_arrets_m':FUSION_ARRETS_M,'fusion_stations_m':FUSION_STATIONS_M,
        'seuil_diffusion':SEUIL_DIFFUSION,'poids_free_floating_par_mode':POIDS_FREE_FLOATING_PAR_MODE,
        'distance':'vol d oiseau (V1, borne haute)','silver_run_id':RUN_ID,
        'date_calcul_utc':dt.datetime.now(dt.timezone.utc).isoformat()}
params.update({'poids_'+k_:v for k_,v in POIDS.items()})
write_table(GOLD,'gold_acces_parametres',pd.DataFrame([{'parametre':a,'valeur':str(b)} for a,b in params.items()]))
write_table(GOLD,'gold_acces_qualite',pd.DataFrame(STATUS))
print('Accès prêt. France entière, V1 vol d oiseau : borne haute.')